# Détection d'anomalies routières — Étape 4 : Benchmarking

**Notebook :** `04_benchmark.ipynb`

Toutes les approches de l'étape 3 ont produit leurs prédictions **dans le même format** : pour
chaque image, une carte de score 960 × 540 (0 à 255), calculée par le modèle du tour où cette image
était en **test**, et un seuil choisi sur la **validation** de ce tour. Chaque image est donc
évaluée exactement une fois par chaque modèle, sans jamais avoir été vue par lui à
l'entraînement.

Ce notebook ne réentraîne rien : il **compare** tous les modèles sur les mêmes 60 images, avec les
mêmes métriques.

| § | Question |
|---|---|
| 0 | Chargement des prédictions de 03a, 03b, 03c |
| 1 | Tableau général : quel modèle est le meilleur ? |
| 2 | Les écarts sont-ils significatifs ? (test statistique) |
| 3 | Résultats par catégorie d'anomalie |
| 4 | Résultats par taille d'objet (détection objet par objet) |
| 5 | Généralisation : les pneus jamais vus (tour 3) |
| 6 | Fausses alarmes : où sont-elles ? (raccourci de position) |
| 7 | Comparaison visuelle |
| 8 | Une piste : combiner deux modèles |
| 9 | Choix du modèle à déployer |

**GPU : inutile** pour ce notebook.

## 0. Chargement

Sélectionnez **en une seule fois** (Ctrl + clic) :

- `RoadAnomaly.zip` et `roadanomaly_lib.py` ;
- `resultats_etape3.zip` (03a), `resultats_etape3b.zip` (03b), `resultats_etape3c.zip` (03c).

Une archive manquante n'empêche pas le notebook de tourner : les modèles correspondants sont
simplement absents des tableaux.

In [ ]:
import os, sys, zipfile, shutil, json, re
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BASE = Path("/content") if IN_COLAB else Path.cwd()
RUNS = BASE / "etape3_runs"; RUNS.mkdir(exist_ok=True)
OUT = BASE / "resultats_etape4"; (OUT / "figures").mkdir(parents=True, exist_ok=True)

def trouver_dataset(dossier):
    dossier = Path(dossier)
    hits = sorted(dossier.rglob("frame_list.json")) if dossier.exists() else []
    return hits[0].parent if hits else None

if IN_COLAB:
    os.chdir(BASE)
    if trouver_dataset(BASE / "data") is None or not any(RUNS.iterdir()):
        from google.colab import files
        print("Sélectionnez RoadAnomaly.zip, roadanomaly_lib.py et les archives resultats_etape3*.zip :")
        for nom in files.upload():
            p = BASE / nom
            if nom.startswith("RoadAnomaly") and nom.endswith(".zip"):
                with zipfile.ZipFile(p) as z: z.extractall(BASE / "data")
                os.remove(p)
            elif nom.startswith("resultats_etape3") and nom.endswith(".zip"):
                cle = re.match(r"resultats_etape3[abc]?", nom).group(0)
                with zipfile.ZipFile(p) as z: z.extractall(RUNS / cle)
                os.remove(p)
            elif nom.startswith("roadanomaly_lib") and nom.endswith(".py") and nom != "roadanomaly_lib.py":
                shutil.move(p, BASE / "roadanomaly_lib.py")
    DATA_ROOT = trouver_dataset(BASE / "data")
else:
    DATA_ROOT = next((p for p in [Path("RoadAnomaly"), Path("../RoadAnomaly")]
                      if (p / "frame_list.json").exists()), None)

assert DATA_ROOT is not None, "Dataset introuvable : envoyez RoadAnomaly.zip"
assert (BASE / "roadanomaly_lib.py").exists(), "roadanomaly_lib.py manquant (dans resultats_etape2.zip)"

import importlib, numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
try:
    import albumentations
    assert int(albumentations.__version__.split(".")[0]) >= 2
except (ImportError, AssertionError):
    %pip install -q "albumentations==2.0.8"
    importlib.invalidate_caches()
sys.path.insert(0, str(BASE))
import roadanomaly_lib; importlib.reload(roadanomaly_lib)
from roadanomaly_lib import *
from scipy.stats import wilcoxon

splits = make_splits(DATA_ROOT)
frames = splits["name"].tolist()
print("Dataset :", DATA_ROOT)
print("Archives trouvées :", sorted(p.name for p in RUNS.iterdir()))
plt.rcParams.update({"figure.dpi": 80, "savefig.dpi": 150, "savefig.bbox": "tight",
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e4e3df", "axes.axisbelow": True})

In [ ]:
# Catalogue des modèles : (étiquette, famille)
CATALOG = {
    "A_unet_scratch":  ("A. U-Net de zéro", "supervisé"),
    "B_unet_imagenet": ("B. U-Net pré-entraîné", "supervisé"),
    "C_deeplabv3plus": ("C. DeepLabV3+ pré-entraîné", "supervisé"),
    "D_ood_msp":       ("D1. Cityscapes + MSP", "sans masque"),
    "D_ood_maxlogit":  ("D2. Cityscapes + max-logit", "sans masque"),
    "D_ood_entropy":   ("D3. Cityscapes + entropie", "sans masque"),
    "E_ae_l1":         ("E1. Autoencodeur L1", "reconstruction"),
    "E_ae_ssim":       ("E2. Autoencodeur SSIM", "reconstruction"),
}
FAMILY_COLOR = {"supervisé": "#2a78d6", "sans masque": "#eb6834", "reconstruction": "#8a8984"}

MASKS = {n: cv2.resize(load_mask(DATA_ROOT, n), (WORK_W, WORK_H), interpolation=cv2.INTER_NEAREST).astype(bool)
         for n in frames}
FOLD_OF = dict(zip(splits.name, splits.fold))

P, THR, SUMM = {}, {}, []          # P[modèle][image] = carte uint8 ; THR[modèle][tour] = seuil 0..255
for key in CATALOG:
    files_ = sorted(RUNS.rglob(f"preds/{key}_fold*.npz"))
    if not files_:
        continue
    P[key], THR[key] = {}, {}
    for f in files_:
        k = int(re.search(r"fold(\d)", f.name).group(1))
        P[key].update(dict(np.load(f)))
        s = json.load(open(f.parent.parent / f"summary_{key}_fold{k}.json"))
        THR[key][k] = int(round(s["threshold"] * 255)); SUMM.append(s)
    manquantes = set(frames) - set(P[key])
    print(f"{CATALOG[key][0]:<30} {len(P[key]):2d} images" + (f"  ({len(manquantes)} manquantes)" if manquantes else ""))

MODELS = list(P)
COMPLETE = [m for m in MODELS if len(P[m]) == len(frames)]
SUMM = pd.DataFrame(SUMM)
LAB = {m: CATALOG[m][0] for m in MODELS}
assert COMPLETE, "Aucun modèle n'a de prédictions pour les 60 images : vérifiez les archives"

## 1. Tableau général

Deux familles de métriques, calculées sur les **60 images** (tous les pixels réunis) :

- **sans seuil** — AUROC, **AP** (précision moyenne, aire sous la courbe précision-rappel) et
  FPR95 : la qualité de la carte de score elle-même. **L'AP est la métrique principale** du
  benchmark RoadAnomaly, car elle tient compte de la rareté des anomalies (hasard ≈ 0,10) ;
- **au seuil choisi sur la validation** — précision, rappel, F1 (= Dice), IoU : la qualité de la
  décision finale « anomalie / pas anomalie ».

Deux façons de moyenner l'IoU :

- **IoU globale** : tous les pixels de test réunis. Les grandes anomalies pèsent plus ;
- **IoU moyenne par image** : chaque image compte autant. Les petites anomalies (cônes, pneus)
  pèsent autant que les grandes.

In [ ]:
def hist_of(model, names):
    pos, neg = np.zeros(256, np.int64), np.zeros(256, np.int64)
    for n in names:
        s, m = P[model][n], MASKS[n]
        pos += np.bincount(s[m], minlength=256); neg += np.bincount(s[~m], minlength=256)
    return pos, neg

def curve_metrics(pos, neg):
    tp = pos[::-1].cumsum(); fp = neg[::-1].cumsum()
    tpr = np.r_[0, tp / pos.sum()]; fpr = np.r_[0, fp / neg.sum()]
    auroc = np.trapezoid(tpr, fpr) if hasattr(np, "trapezoid") else np.trapz(tpr, fpr)
    ap = np.sum(np.diff(np.r_[0, tp / pos.sum()]) * tp / np.maximum(tp + fp, 1))
    return dict(AUROC=auroc, AP=ap, FPR95=fpr[np.argmax(tpr >= 0.95)])

def counts(model, n):
    # TP, FP, FN de l'image n au seuil du tour où elle était en test
    pred = P[model][n] >= THR[model][FOLD_OF[n]]; m = MASKS[n]
    return np.array([(pred & m).sum(), (pred & ~m).sum(), (~pred & m).sum()], np.int64)

C = {m: {n: counts(m, n) for n in P[m]} for m in MODELS}          # TP, FP, FN par image
IOU_IMG = {m: {n: c[0] / max(c.sum(), 1) for n, c in C[m].items()} for m in MODELS}

rows = []
for m in COMPLETE:
    tp, fp, fn = np.sum(list(C[m].values()), axis=0)
    cm = curve_metrics(*hist_of(m, frames))
    s = SUMM[SUMM.model == m]
    rows.append({"modèle": LAB[m], "famille": CATALOG[m][1], **cm,
                 "précision": tp / max(tp + fp, 1), "rappel": tp / max(tp + fn, 1),
                 "F1 (= Dice)": 2 * tp / max(2 * tp + fp + fn, 1), "IoU globale": tp / max(tp + fp + fn, 1),
                 "IoU par image": np.mean(list(IOU_IMG[m].values())),
                 "IoU tours (moy.)": s.test_iou.mean(), "IoU tours (é.-t.)": s.test_iou.std(),
                 "key": m})
BENCH = pd.DataFrame(rows).sort_values("AP", ascending=False).reset_index(drop=True)
show = BENCH.drop(columns="key").set_index("modèle")
display(show.style.format({c: "{:.3f}" for c in show.columns if c != "famille"})
            .background_gradient(subset=["AP", "IoU globale"], cmap="Blues"))
BENCH.to_csv(OUT / "benchmark_general.csv", index=False)
print(f"Référence : un détecteur au hasard a une AP de {np.mean([m.mean() for m in MASKS.values()]):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 0.45 * len(BENCH) + 1.4), sharey=True)
order = BENCH.iloc[::-1]
for ax, col, titre in [(axes[0], "AP", "AP (sans seuil)"), (axes[1], "IoU globale", "IoU (au seuil de validation)")]:
    bars = ax.barh(order["modèle"], order[col], color=[FAMILY_COLOR[f] for f in order["famille"]], height=0.62)
    for b, v in zip(bars, order[col]):
        ax.text(v + 0.01, b.get_y() + b.get_height() / 2, f"{v:.3f}", va="center", fontsize=9)
    ax.set_xlim(0, 1); ax.set_title(titre, fontweight="bold"); ax.grid(axis="y", visible=False)
base = np.mean([m.mean() for m in MASKS.values()])
axes[0].axvline(base, color="#52514e", ls="--", lw=1); axes[0].text(base + 0.01, len(BENCH) - 0.55, "hasard", fontsize=8, color="#52514e")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in FAMILY_COLOR.values()]
fig.legend(handles, FAMILY_COLOR.keys(), loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Classement des modèles sur les 60 images", fontweight="bold")
plt.tight_layout(rect=(0, 0.05, 1, 1)); fig.savefig(OUT / "figures" / "01_classement_modeles.png"); plt.show()

## 2. Les écarts sont-ils significatifs ?

Avec 60 images, un écart de 0,01 d'IoU peut être dû au hasard. On utilise le **test de Wilcoxon
pour échantillons appariés** sur l'IoU **par image** : pour chaque image, on compare les deux
modèles, puis on teste si l'un gagne systématiquement.

- Il est **apparié** : les deux modèles sont évalués sur les mêmes images, ce qui élimine l'effet
  « image facile / image difficile ».
- Il est **non paramétrique** : il ne suppose pas que les IoU suivent une loi normale (elles ne le
  font pas : beaucoup sont proches de 0 ou de 1).

**Lecture :** une *p*-valeur < 0,05 signifie que l'écart a moins de 5 % de chances d'être dû au
hasard ; on le considère alors comme **significatif**.

In [ ]:
def compare(m1, m2):
    a = np.array([IOU_IMG[m1][n] for n in frames]); b = np.array([IOU_IMG[m2][n] for n in frames])
    p = wilcoxon(a, b).pvalue if np.any(a != b) else 1.0
    return {"comparaison": f"{LAB[m1]}  vs  {LAB[m2]}", "IoU/image modèle 1": a.mean(),
            "IoU/image modèle 2": b.mean(), "images où 1 gagne": int((a > b).sum()),
            "images où 2 gagne": int((b > a).sum()), "p-valeur": p,
            "significatif (p < 0,05)": "oui" if p < 0.05 else "non"}

paires = [("A_unet_scratch", "B_unet_imagenet", "apport du pré-entraînement"),
          ("B_unet_imagenet", "C_deeplabv3plus", "U-Net vs DeepLabV3+"),
          ("C_deeplabv3plus", "D_ood_maxlogit", "supervisé vs sans masque"),
          ("D_ood_maxlogit", "E_ae_l1", "Cityscapes vs autoencodeur")]
stats = pd.DataFrame([{"question": q, **compare(a, b)} for a, b, q in paires if a in COMPLETE and b in COMPLETE])
display(stats.style.format({"IoU/image modèle 1": "{:.3f}", "IoU/image modèle 2": "{:.3f}", "p-valeur": "{:.4f}"}))
stats.to_csv(OUT / "tests_statistiques.csv", index=False)

## 3. Résultats par catégorie

L'IoU globale mélange des catégories très différentes. Ici, chaque catégorie est évaluée sur ses
seules images (pixels réunis, au seuil de validation). Rappel : il n'y a que **3 images** de pneus
et **3** de véhicules ; leurs scores sont donc fragiles.

In [ ]:
CAT_FR = {"animals": "animaux", "obstacles": "obstacles", "cones": "cônes", "lost_tires": "pneus perdus", "vehicle": "véhicules"}
cat_rows, cat_ap = [], []
for m in COMPLETE:
    r, ra = {"modèle": LAB[m]}, {"modèle": LAB[m]}
    for c in CATEGORIES:
        names = [n for n in frames if category_of(n) == c]
        tp, fp, fn = np.sum([C[m][n] for n in names], axis=0)
        r[CAT_FR[c]] = tp / max(tp + fp + fn, 1)
        ra[CAT_FR[c]] = curve_metrics(*hist_of(m, names))["AP"]
    cat_rows.append(r); cat_ap.append(ra)
cat_iou = pd.DataFrame(cat_rows).set_index("modèle").loc[[LAB[m] for m in BENCH.key]]
cat_ap = pd.DataFrame(cat_ap).set_index("modèle").loc[[LAB[m] for m in BENCH.key]]
print("IoU par catégorie"); display(cat_iou.style.format("{:.3f}").background_gradient(cmap="Blues", axis=None, vmin=0, vmax=1))
print("AP par catégorie");  display(cat_ap.style.format("{:.3f}").background_gradient(cmap="Blues", axis=None, vmin=0, vmax=1))
cat_iou.to_csv(OUT / "iou_par_categorie.csv"); cat_ap.to_csv(OUT / "ap_par_categorie.csv")

## 4. Résultats par taille d'objet

Pour un véhicule, ce qui compte est souvent : **chaque objet a-t-il été vu ?** On évalue donc
objet par objet :

- un objet = une composante connexe du masque (au moins 100 px en pleine résolution, comme à
  l'étape 1) ;
- il est **détecté** si au moins **50 % de ses pixels** sont signalés comme anomalie ;
- tailles selon la convention COCO (aire en pleine résolution) : petit < 32², moyen 32²–96²,
  grand > 96².

In [ ]:
SCALE_AREA = (1280 * 720) / (WORK_W * WORK_H)          # aire 960x540 -> pleine résolution
objs = []
for n in frames:
    k_, lab, st, _ = cv2.connectedComponentsWithStats(MASKS[n].astype(np.uint8), connectivity=8)
    for i in range(1, k_):
        area_full = st[i, cv2.CC_STAT_AREA] * SCALE_AREA
        if area_full < 100:
            continue
        comp = lab == i
        o = {"name": n, "category": category_of(n), "area": area_full,
             "taille": "petit" if area_full < 32 ** 2 else ("moyen" if area_full < 96 ** 2 else "grand")}
        for m in COMPLETE:
            o[m] = (P[m][n][comp] >= THR[m][FOLD_OF[n]]).mean() >= 0.5
        objs.append(o)
objs = pd.DataFrame(objs)
print(f"{len(objs)} objets :", objs.taille.value_counts().reindex(["petit", "moyen", "grand"]).to_dict())

det = pd.DataFrame({LAB[m]: objs.groupby("taille")[m].mean() for m in COMPLETE}).T[["petit", "moyen", "grand"]]
det["tous"] = [objs[m].mean() for m in COMPLETE]
det = det.loc[[LAB[m] for m in BENCH.key]]
print("Taux de détection des objets (part des objets dont ≥ 50 % des pixels sont détectés)")
display(det.style.format("{:.0%}").background_gradient(cmap="Blues", axis=None, vmin=0, vmax=1))
det.to_csv(OUT / "detection_par_taille.csv")

In [ ]:
show_m = [m for m in ["A_unet_scratch", "B_unet_imagenet", "C_deeplabv3plus", "D_ood_maxlogit", "E_ae_l1"] if m in COMPLETE]
sizes = ["petit", "moyen", "grand"]
fig, ax = plt.subplots(figsize=(10, 4.4))
w = 0.8 / len(show_m)
shades = {"A_unet_scratch": ("#2a78d6", 0.35), "B_unet_imagenet": ("#2a78d6", 0.7), "C_deeplabv3plus": ("#2a78d6", 1.0),
          "D_ood_maxlogit": ("#eb6834", 1.0), "E_ae_l1": ("#8a8984", 1.0)}
for i, m in enumerate(show_m):
    v = [objs.loc[objs.taille == s, m].mean() for s in sizes]
    col, alpha = shades[m]
    bars = ax.bar(np.arange(3) + (i - (len(show_m) - 1) / 2) * w, v, width=w * 0.92, color=col, alpha=alpha, label=LAB[m])
    for b, val in zip(bars, v):
        ax.text(b.get_x() + b.get_width() / 2, val + 0.015, f"{val:.0%}", ha="center", fontsize=7.5)
n_by = objs.taille.value_counts()
ax.set_xticks(range(3)); ax.set_xticklabels([f"{s} ({n_by.get(s, 0)} objets)" for s in sizes])
ax.set_ylim(0, 1.1); ax.set_ylabel("Objets détectés"); ax.grid(axis="x", visible=False)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Taux de détection des objets selon leur taille", fontweight="bold")
ax.legend(frameon=False, fontsize=8, ncol=len(show_m), loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.tight_layout(); fig.savefig(OUT / "figures" / "02_detection_par_taille.png"); plt.show()

## 5. Généralisation : les pneus jamais vus

Les 3 images de pneus viennent d'une seule vidéo et sont toutes dans le pli 3. Quand elles sont
testées (tour 3), les modèles supervisés **n'ont jamais vu de pneu** à l'entraînement. C'est le
seul test, dans ce dataset, d'une anomalie **d'un type nouveau**, ce qui est la situation réelle
d'un véhicule autonome. Les modèles D et E ne voient jamais d'anomalie ; pour eux, rien ne change.

In [ ]:
tyres = [n for n in frames if category_of(n) == "lost_tires"]
rows = []
for m in COMPLETE:
    o = objs[objs.category == "lost_tires"]
    tp, fp, fn = np.sum([C[m][n] for n in tyres], axis=0)
    rows.append({"modèle": LAB[m], "pneus détectés": f"{int(o[m].sum())} / {len(o)}",
                 "rappel pixels": tp / max(tp + fn, 1), "IoU": tp / max(tp + fp + fn, 1),
                 "AP": curve_metrics(*hist_of(m, tyres))["AP"]})
tyre_tab = pd.DataFrame(rows).set_index("modèle").loc[[LAB[m] for m in BENCH.key]]
display(tyre_tab.style.format({"rappel pixels": "{:.3f}", "IoU": "{:.3f}", "AP": "{:.3f}"}))
tyre_tab.to_csv(OUT / "generalisation_pneus.csv")

## 6. Où sont les fausses alarmes ?

L'étape 1 avait signalé un **risque de raccourci** : les anomalies étant presque toujours dans
une bande sous l'horizon, un modèle pourrait apprendre « cette zone = anomalie ». On compare le
profil vertical des **fausses alarmes** de chaque modèle à celui des vraies anomalies : si les
fausses alarmes suivent exactement le même profil, le modèle s'appuie sans doute sur la position.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
gt = np.mean([MASKS[n] for n in frames], axis=0).mean(1)
ax.plot(gt / gt.sum(), np.arange(WORK_H), color="#2b2b2b", lw=2, ls="--", label="vraies anomalies")
fp_share = {}
for m in show_m:
    fpm = np.mean([(P[m][n] >= THR[m][FOLD_OF[n]]) & ~MASKS[n] for n in frames], axis=0)
    prof = fpm.mean(1)
    col, alpha = shades[m]
    ax.plot(prof / max(prof.sum(), 1e-9), np.arange(WORK_H), color=col, alpha=alpha, lw=2, label=LAB[m])
    fp_share[LAB[m]] = fpm.mean()
ax.set_ylim(WORK_H, 0); ax.set_xlim(left=0)
ax.set_ylabel("Ligne de l'image (0 = haut)"); ax.set_xlabel("Répartition verticale (normalisée)")
ax.set_title("Profil vertical des fausses alarmes vs des vraies anomalies", fontweight="bold")
ax.legend(frameon=False, fontsize=8, loc="lower right")
plt.tight_layout(); fig.savefig(OUT / "figures" / "03_profil_fausses_alarmes.png"); plt.show()
print("Part de l'image signalée à tort (fausses alarmes / pixels de l'image) :")
for k_, v in fp_share.items(): print(f"  {k_:<30} {v:.2%}")

## 7. Comparaison visuelle

Pour chaque image, la décision de chaque modèle à son seuil de validation :
**bleu = anomalie trouvée** (vrai positif), **rouge = fausse alarme** (faux positif),
**jaune = anomalie manquée** (faux négatif).

In [ ]:
choix = ["animals22_red_fox_on_road", "obstacles09_boulder3", "cones03_Bang_Na_Trat_Highway",
         "lost_tires08", "vehicle01_man_carrying_dry_grass_on_bicycle", "animals05_cows_near_Phonsavan_Laos"]
choix = [n for n in choix if n in MASKS]
def overlay3(img, pred, m):
    out = (img * 0.55).astype(np.uint8)
    out[pred & m] = (42, 120, 214); out[pred & ~m] = (227, 73, 72); out[~pred & m] = (237, 190, 0)
    return out
fig, axes = plt.subplots(len(choix), len(show_m) + 1, figsize=(2.9 * (len(show_m) + 1), 1.7 * len(choix)))
for r, n in enumerate(choix):
    img = cv2.resize(load_image(DATA_ROOT, n), (WORK_W, WORK_H), interpolation=cv2.INTER_AREA)
    axes[r, 0].imshow(img); axes[r, 0].set_ylabel(n[:18], fontsize=7)
    for c, m in enumerate(show_m, start=1):
        pred = P[m][n] >= THR[m][FOLD_OF[n]]
        axes[r, c].imshow(overlay3(img, pred, MASKS[n]))
        axes[r, c].set_xlabel(f"IoU {IOU_IMG[m][n]:.2f}", fontsize=8, labelpad=2)
        if r == 0: axes[r, c].set_title(LAB[m], fontsize=8)
    if r == 0: axes[r, 0].set_title("Image", fontsize=8)
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.tight_layout(); fig.savefig(OUT / "figures" / "04_comparaison_visuelle.png"); plt.show()

## 8. Une piste : combiner deux modèles

Deux modèles qui ne font pas les mêmes erreurs peuvent se compléter. On teste deux combinaisons
simples, en faisant la **moyenne des scores** (après conversion en rangs centiles pour D, dont
l'échelle est différente) :

- **B + C** : deux modèles supervisés aux architectures différentes (ensemble classique) ;
- **C + D2** : un modèle qui a appris nos anomalies et un modèle qui détecte l'inconnu.

N'ayant pas de seuil de validation pour ces combinaisons, on compare uniquement les métriques
**sans seuil** (AUROC, AP).

In [ ]:
def to_rank(model):
    # convertit les scores 0..255 d'un modèle en rangs centiles (0..1) sur l'ensemble des pixels
    pos, neg = hist_of(model, frames); cdf = np.cumsum(pos + neg) / (pos.sum() + neg.sum())
    return {n: cdf[P[model][n]] for n in frames}

def ensemble_metrics(models):
    R = [to_rank(m) for m in models]
    pos, neg = np.zeros(256, np.int64), np.zeros(256, np.int64)
    for n in frames:
        s = (np.mean([r[n] for r in R], axis=0) * 255).round().astype(np.uint8); m = MASKS[n]
        pos += np.bincount(s[m], minlength=256); neg += np.bincount(s[~m], minlength=256)
    return curve_metrics(pos, neg)

ens_rows = []
for combo in [("B_unet_imagenet", "C_deeplabv3plus"), ("C_deeplabv3plus", "D_ood_maxlogit")]:
    if all(m in COMPLETE for m in combo):
        for m in combo:
            ens_rows.append({"modèle": LAB[m], **curve_metrics(*hist_of(m, frames))})
        ens_rows.append({"modèle": " + ".join(LAB[m].split(".")[0] for m in combo) + " (moyenne)", **ensemble_metrics(combo)})
ens = pd.DataFrame(ens_rows).drop_duplicates("modèle").set_index("modèle")
display(ens.style.format("{:.3f}"))
ens.to_csv(OUT / "ensembles.csv")

## 9. Choix du modèle à déployer

**Critère principal : l'AP** (métrique de référence du benchmark, indépendante du seuil), puis
l'IoU. Si deux modèles ne sont pas significativement différents (section 2), on départage par la
**stabilité** entre les tours (écart-type plus faible) puis par la **légèreté** (moins de
paramètres, inférence plus rapide pour une application).

Le choix n'est retenu que parmi des modèles **déployables seuls** (pas d'ensemble, à ce stade).

In [ ]:
PARAMS_M = {"A_unet_scratch": 24.4, "B_unet_imagenet": 24.4, "C_deeplabv3plus": 22.4,
            "D_ood_msp": 85.0, "D_ood_maxlogit": 85.0, "D_ood_entropy": 85.0, "E_ae_l1": 1.6, "E_ae_ssim": 1.6}
cand = BENCH.copy()
cand["paramètres (M)"] = cand.key.map(PARAMS_M)
top = cand.iloc[0]
proches = cand[(top["AP"] - cand["AP"]) < 0.02]          # à moins de 0,02 d'AP du meilleur
if len(proches) > 1:
    choix_final = proches.sort_values(["IoU tours (é.-t.)", "paramètres (M)"]).iloc[0]
    raison = (f"{len(proches)} modèles à moins de 0,02 d'AP du meilleur ; départagés par la stabilité "
              f"(écart-type d'IoU entre tours) puis le nombre de paramètres")
else:
    choix_final = top; raison = "meilleure AP, avec une avance nette"

display(cand[["modèle", "AP", "IoU globale", "IoU tours (moy.)", "IoU tours (é.-t.)", "paramètres (M)"]]
        .head(4).set_index("modèle").style.format("{:.3f}"))
print(f"\nModèle retenu pour le déploiement : {choix_final['modèle']}")
print("Raison :", raison)

ARCH = {"A_unet_scratch": "Unet", "B_unet_imagenet": "Unet", "C_deeplabv3plus": "DeepLabV3Plus"}
best = {"key": choix_final["key"], "label": choix_final["modèle"], "AP": float(choix_final["AP"]),
        "IoU_globale": float(choix_final["IoU globale"]),
        "IoU_tours_moyenne": float(choix_final["IoU tours (moy.)"]),
        "IoU_tours_ecart_type": float(choix_final["IoU tours (é.-t.)"]),
        "seuil_moyen": float(SUMM[SUMM.model == choix_final["key"]].threshold.mean()),
        "meilleure_epoque_moyenne": float(SUMM[SUMM.model == choix_final["key"]].best_epoch.mean()),
        "architecture": ARCH.get(choix_final["key"]), "encodeur": "resnet34", "raison": raison}
json.dump(best, open(OUT / "modele_retenu.json", "w"), indent=1, ensure_ascii=False)
print(json.dumps(best, indent=1, ensure_ascii=False))

### Comment rédiger la conclusion du benchmark

Appuyez-vous sur vos tableaux pour répondre à ces questions dans le rapport :

1. **Le classement** (section 1) : quelle famille domine, et de combien ? Les modèles supervisés
   pré-entraînés devraient nettement devancer les autres en AP et en IoU.
2. **La significativité** (section 2) : l'écart entre U-Net et DeepLabV3+ est-il significatif ?
   S'il ne l'est pas, dites-le explicitement : « les deux architectures sont équivalentes sur ce
   dataset ».
3. **Les points faibles** (sections 3 et 4) : quelles catégories et quelles tailles d'objets
   restent difficiles pour **tous** les modèles ? Ce sont les pistes d'amélioration.
4. **La généralisation** (section 5) : un modèle supervisé détecte-t-il des pneus sans en avoir
   jamais vu ? C'est l'argument pour ou contre l'approche sans masque en conditions réelles.
5. **Le raccourci de position** (section 6) : les fausses alarmes suivent-elles le profil des
   vraies anomalies ?
6. **La combinaison** (section 8) : apporte-t-elle un gain ? Si oui, c'est une amélioration
   concrète à proposer.
7. **Le choix** (section 9) : quel modèle est déployé, et pourquoi ?

**Étape suivante : `05_deploiement.ipynb`.** Le modèle retenu est réentraîné sur **toutes** les
images (le nombre d'époques et le seuil viennent de `modele_retenu.json`), exporté, puis intégré à
une petite application web.

In [ ]:
shutil.make_archive(str(BASE / "resultats_etape4"), "zip", OUT)
if IN_COLAB:
    from google.colab import files
    files.download(str(BASE / "resultats_etape4.zip"))
print(sorted(str(p.relative_to(OUT)) for p in OUT.rglob("*") if p.is_file()))